In [5]:
import numpy as np
import networkx as nx
import pickle
import os
import json

from utils import *

%matplotlib inline
%load_ext autoreload
%autoreload 2

setup_environment()
set_paper_style()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [1]:
import json
import networkx as nx
import plotly.graph_objects as go
import pandas as pd

# ==============================================================================
# 設定エリア
# ==============================================================================
JSON_FILE_PATH = "knowledge_graph_enhanced.json"
# JSON_FILE_PATH = "knowledge_graph_with_wiki.json"
TARGET_PREFECTURE = "愛知県"
# ==============================================================================


# 1. JSONファイルの読み込みとNetworkXグラフの構築
print(f"🔄 '{JSON_FILE_PATH}' を読み込んでいます...")
with open(JSON_FILE_PATH, "r", encoding="utf-8") as f:
    graph_data = json.load(f)

G = nx.Graph()
for node_data in graph_data["nodes"]:
    attributes = node_data.get("properties", {})
    attributes["type"] = node_data.get("type")
    attributes["label"] = node_data.get("label")
    G.add_node(node_data["id"], **attributes)

for edge_data in graph_data["edges"]:
    G.add_edge(edge_data["source"], edge_data["target"], type=edge_data.get("label"))
print("✅ グラフの読み込みが完了しました。")

# 2. 指定された都道府県のサブグラフを作成
print(f"🔄 '{TARGET_PREFECTURE}' のサブグラフを作成しています...")
subgraph_nodes = {TARGET_PREFECTURE}
for node, data in G.nodes(data=True):
    if data.get("都道府県") == TARGET_PREFECTURE:
        subgraph_nodes.add(node)
H = G.subgraph(subgraph_nodes)

# 3. グラフのレイアウト計算
# spring_layoutアルゴリズムでノードの2D座標を計算
print("🔄 グラフのレイアウトを計算しています...")
pos = nx.spring_layout(H, seed=42)  # seedを固定して毎回同じレイアウトにする

# 4. Plotly用のデータを作成
# 4.1. エッジのトレースを作成
edge_x = []
edge_y = []
for edge in H.edges():
    x0, y0 = pos[edge[0]]
    x1, y1 = pos[edge[1]]
    edge_x.extend([x0, x1, None])
    edge_y.extend([y0, y1, None])

edge_trace = go.Scatter(
    x=edge_x,
    y=edge_y,
    line=dict(width=0.5, color="#888"),
    hoverinfo="none",
    mode="lines",
)

# 4.2. ノードのトレースを作成
node_x = []
node_y = []
node_text = []
node_color = []
node_size = []
node_hover_text = []

for node_id, attrs in H.nodes(data=True):
    x, y = pos[node_id]
    node_x.append(x)
    node_y.append(y)
    node_text.append(attrs.get("label", ""))

    # ツールチップ（ホバーテキスト）の情報を生成
    title_html = f"<b>{attrs.get('label', node_id)}</b><br>Type: {attrs.get('type')}"
    if "properties" in attrs:
        for key, val in attrs["properties"].items():
            if isinstance(val, dict) and "value" in val:
                display_val = f"{val['value']} (year: {val['year']})"
                title_html += f"<br><b>{key}</b>: {display_val}"
            else:
                title_html += f"<br><b>{key}</b>: {val}"
    node_hover_text.append(title_html)

    # ノードの種類によって色とサイズを変更
    if attrs.get("type") == "都道府県":
        node_color.append("#ff6961")
        node_size.append(30)
    else:
        node_color.append("#77dd77")
        node_size.append(15)

node_trace = go.Scatter(
    x=node_x,
    y=node_y,
    mode="markers+text",
    text=node_text,
    textposition="bottom center",
    hoverinfo="text",
    hovertext=node_hover_text,
    marker=dict(showscale=False, color=node_color, size=node_size, line_width=2),
)

# 5. Figureを作成して表示
print("🔄 Plotlyでグラフを描画しています...")
fig = go.Figure(
    data=[edge_trace, node_trace],
    layout=go.Layout(
        title=f"<br>知識グラフ: {TARGET_PREFECTURE}",
        titlefont_size=16,
        showlegend=False,
        hovermode="closest",
        margin=dict(b=20, l=5, r=5, t=40),
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    ),
)

fig.show()
print("🎉 完了！ グラフが以下に表示されます。")

🔄 'knowledge_graph_enhanced.json' を読み込んでいます...
✅ グラフの読み込みが完了しました。
🔄 '愛知県' のサブグラフを作成しています...
🔄 グラフのレイアウトを計算しています...
🔄 Plotlyでグラフを描画しています...


🎉 完了！ グラフが以下に表示されます。
